In [1]:
"""
=============================================================================
HAN-QKV Cyber Attack Detection — Unified Gradio UI  v2
=============================================================================
Fixes:
  - BytesIO → PIL Image (fig2pil now returns PIL.Image)
  - Steps auto-advance: completing one step reveals the next
  - Single-page accordion layout (no tabs) — clean sequential flow
=============================================================================
Run:
    pip install gradio torch scikit-learn pandas matplotlib seaborn pillow
    python han_qkv_ui_v2.py
    Open  http://127.0.0.1:7860
=============================================================================
"""

import io, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image                          # ← fix: return PIL not BytesIO

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

import gradio as gr

warnings.filterwarnings("ignore")
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#dddddd",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black"
})

# ─── Pipeline state ───────────────────────────────────────────────────────────
S = {
    "dataset": None,
    "df": None,
    "X_scaled": None, "y": None,
    "X_final": None,  "y_final": None,
    "X_train": None,  "X_test": None,
    "y_train": None,  "y_test": None,
    "X_train_t": None,"X_test_t": None,
    "y_train_t": None,"y_test_t": None,
    "class_weights": None,
    "model": None, "optimizer": None, "scheduler": None,
    "target_encoder": None, "scaler": None,
    "_X_raw": None, "_y_raw": None,
    "train_losses": [], "val_accs": [],
    "y_true": None,  "y_pred": None,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}

# ─── Colours ──────────────────────────────────────────────────────────────────
DARK  = "#dddddd"; CARD  = "#161b22"; BORDER = "#dddddd"
BLUE  = "#58a6ff"; CYAN  = "#00b4d8"; PINK   = "#f72585"
GREEN = "#3fb950"
PAL   = [CYAN,PINK,GREEN,"#4cc9f0","#7209b7","#f4a261","#2dc653","#e76f51","#a8dadc","#f1c40f"]

# ─── Helpers ──────────────────────────────────────────────────────────────────
def fig2pil(fig) -> Image.Image:
    """Convert matplotlib figure → PIL Image (what Gradio needs)."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    buf.seek(0)
    img = Image.open(buf).copy()   # copy so buffer can be closed
    plt.close(fig)
    return img

def light_fig(*args, **kwargs):
    fig = plt.figure(*args, **kwargs)
    fig.patch.set_facecolor("white")   # ← white background
    return fig

def light_ax(ax):
    ax.set_facecolor("white")          # ← white plot area
    ax.tick_params(colors="black", labelsize=8)

    for sp in ax.spines.values():
        sp.set_edgecolor("#dddddd")    # light border

    return ax
    
def err(msg): return f"⚠️  {msg}", gr.update(visible=False), gr.update(visible=False)

# ─── Models ───────────────────────────────────────────────────────────────────

class BotMHSA(nn.Module):
    def __init__(self, input_dim, d_model, heads):
        super().__init__()
        self.qkv = nn.Linear(input_dim, d_model*3)
        self.heads = heads; self.d_model = d_model
        self.head_dim = d_model // heads
        self.out = nn.Linear(d_model, d_model)
    def forward(self, x):
        B,T,_ = x.size()
        qkv = self.qkv(x).view(B,T,self.heads,3*self.head_dim).permute(0,2,1,3)
        q,k,v = qkv.chunk(3,dim=-1)
        scores = (q @ k.transpose(-1,-2)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores,dim=-1)
        out = (attn@v).permute(0,2,1,3).contiguous().view(B,T,self.d_model)
        return self.out(out)

class BotQKV_HAN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.gru1  = nn.GRU(input_dim, 64, bidirectional=True, batch_first=True)
        self.attn1 = BotMHSA(128,128,4)
        self.gru2  = nn.GRU(128, 64, bidirectional=True, batch_first=True)
        self.attn2 = BotMHSA(128,128,4)
        self.fc    = nn.Linear(128, num_classes)
    def forward(self, x):
        h,_ = self.gru1(x);  h = self.attn1(h)
        h,_ = self.gru2(h);  h = self.attn2(h)
        return self.fc(h.mean(1))

class TonMHSA(nn.Module):
    def __init__(self, input_dim, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model=d_model; self.num_heads=num_heads
        self.head_dim=d_model//num_heads
        self.qkv=nn.Linear(input_dim,3*d_model)
        self.out=nn.Linear(d_model,d_model)
        self.drop=nn.Dropout(dropout)
    def forward(self, x, mask=None):
        B,L,_ = x.size()
        qkv = self.qkv(x).view(B,L,self.num_heads,3*self.head_dim).permute(0,2,1,3)
        q,k,v = qkv.chunk(3,dim=-1)
        scores = (q @ k.transpose(-1,-2)) / math.sqrt(self.head_dim)
        attn = self.drop(torch.softmax(scores,dim=-1))
        ctx = (attn@v).permute(0,2,1,3).contiguous().view(B,L,self.d_model)
        return self.out(ctx), attn

class WordEncoder(nn.Module):
    def __init__(self, input_dim, hidden, d_model, heads, dropout=0.1):
        super().__init__()
        self.proj   = nn.Linear(input_dim, hidden)
        self.bi_gru = nn.GRU(hidden, hidden, batch_first=True, bidirectional=True)
        self.attn   = TonMHSA(2*hidden, d_model, heads, dropout)
    def forward(self, x):
        h,_ = self.bi_gru(torch.tanh(self.proj(x)))
        h,_ = self.attn(h)
        return h

class SentenceEncoder(nn.Module):
    def __init__(self, input_dim, hidden, d_model, heads, num_classes, dropout=0.1):
        super().__init__()
        self.bi_gru = nn.GRU(input_dim, hidden, batch_first=True, bidirectional=True)
        self.attn   = TonMHSA(2*hidden, d_model, heads, dropout)
        self.drop   = nn.Dropout(dropout)
        self.fc     = nn.Linear(d_model, num_classes)
    def forward(self, x):
        h,_ = self.bi_gru(x)
        h,_ = self.attn(h)
        return self.fc(self.drop(h.mean(1)))

class TonQKV_HAN(nn.Module):
    def __init__(self, F_in, n_cls, dropout=0.25):
        super().__init__()
        self.word = WordEncoder(F_in, 128, 256, 8, dropout)
        self.sent = SentenceEncoder(256, 128, 256, 8, n_cls, dropout)
    def forward(self, x):
        return self.sent(self.word(x))


# =============================================================================
#  STEP FUNCTIONS  (each returns: status_md, [optional image(s)], next_visible)
# =============================================================================

def step1_seeds(dataset_choice):
    ds = "bot" if "BoT" in dataset_choice else "ton"
    S["dataset"] = ds
    seed = 42
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    md = (f"## ✅ Step 1 — Imports & Seed\n\n"
          f"**Dataset:** `{'BoT-IoT' if ds=='bot' else 'TON-IoT'}`  |  "
          f"**Device:** `{S['device']}`  |  **Seed:** `42`\n\n"
          f"{'🚀 GPU (CUDA) detected.' if torch.cuda.is_available() else '💻 CPU mode.'}")
    # reveal step 2
    return md, gr.update(visible=True)


def step2_load(file_obj):
    if S["dataset"] is None:
        return "⚠️  Run Step 1 first.", gr.update(visible=False), gr.update(visible=False)
    if file_obj is None:
        return "⚠️  Please upload your CSV file.", gr.update(visible=False), gr.update(visible=False)

    try:
        path = file_obj.name if hasattr(file_obj, "name") else str(file_obj)
        df   = pd.read_csv(path, low_memory=False)
        file_name = path.replace("\\", "/").split("/")[-1]
    except Exception as e:
        return f"❌ Error reading file: {e}", gr.update(visible=False), gr.update(visible=False)

    S["df"] = df
    ds = S["dataset"]
    target_col = "category" if ds == "bot" else "type"

    has_target   = target_col in df.columns
    n_classes    = df[target_col].nunique() if has_target else 0
    class_counts = df[target_col].value_counts() if has_target else pd.Series()

    if has_target:
        total = len(df)
        class_rows = "\n".join(
            f"| `{c}` | {n:,} | {n/total*100:.1f}% |"
            for c, n in class_counts.items()
        )
        class_table = f"\n\n**Class breakdown:**\n\n| Class | Count | % |\n|---|---|---|\n{class_rows}"
    else:
        class_table = ""

    warning = ""
    if not has_target:
        obj_cols = df.select_dtypes(include="object").columns.tolist()
        warning = (f"\n\n> ❌ **Column `{target_col}` not found.**\n"
                   f"> Text columns in file: `{obj_cols}`")

    md = (f"## ✅ Step 2 — Data Loaded\n\n"
          f"| Property | Value |\n|---|---|\n"
          f"| File | `{file_name}` |\n"
          f"| Rows | `{df.shape[0]:,}` |\n"
          f"| Columns | `{df.shape[1]}` |\n"
          f"| Missing values | `{df.isnull().sum().sum()}` |\n"
          f"| Target column | `{target_col}` ({'✅ found' if has_target else '❌ missing'}) |\n"
          f"| Unique classes | `{n_classes}` |\n"
          f"| Memory | `{df.memory_usage(deep=True).sum()/1e6:.2f} MB` |"
          + class_table + warning)

    return md, gr.update(value=df.head(), visible=True), gr.update(visible=True)


def step3_distribution():
    df = S.get("df")
    if df is None:
        return "⚠️  Run Step 2 first.", None, gr.update(visible=False)

    ds = S["dataset"]
    target_col = "category" if ds=="bot" else "type"
    dist = df[target_col].value_counts()

    fig = light_fig(figsize=(13, 5))
    ax1 = light_ax(fig.add_subplot(121))
    ax2 = light_ax(fig.add_subplot(122))

    bars = ax1.bar(dist.index, dist.values, color=PAL[:len(dist)], edgecolor="#dddddd")
    for b,v in zip(bars,dist.values):
        ax1.text(b.get_x()+b.get_width()/2, b.get_height()+dist.values.max()*0.01,
                 f"{v:,}", ha="center", va="bottom", color="black", fontsize=8)
    ax1.set_title(f"{'BoT-IoT' if ds=='bot' else 'TON-IoT'} — Attack Distribution",
                  color="black", fontsize=12)
    ax1.set_xlabel("Category", color="black"); ax1.set_ylabel("Count", color="black")
    ax1.tick_params(axis="x", rotation=35)

    ax2.pie(dist.values, labels=dist.index, autopct="%1.1f%%", colors=PAL[:len(dist)],
            wedgeprops={"linewidth":1.2,"edgecolor":"#dddddd"},
            textprops={"color":"black","fontsize":8})
    ax2.set_title("Class Proportions", color="black")
    fig.tight_layout()

    total = dist.sum()
    rows = "\n".join(f"| `{c}` | {n:,} | {n/total*100:.1f}% |" for c,n in dist.items())
    md = f"## ✅ Step 3 — Target Distribution\n\n| Class | Count | % |\n|---|---|---|\n{rows}"
    return md, fig2pil(fig), gr.update(visible=True)


def step4_cleaning():
    df = S.get("df")
    if df is None:
        return "⚠️  Run Step 2 first.", gr.update(visible=False)

    ds = S["dataset"]
    # Exact drop list from notebook — 'category'/'type' kept as target
    if ds == "bot":
        drop = ['pkSeqID','stime','ltime','saddr','daddr','subcategory','label']
    else:
        drop = ['src_ip','dst_ip','dns_query','ssl_subject','ssl_issuer',
                'http_uri','http_user_agent','label']

    dropped = [c for c in drop if c in df.columns]
    df2 = df.drop(columns=dropped)
    S["df"] = df2

    target_col = "category" if ds == "bot" else "type"
    feat_cols  = [c for c in df2.columns if c != target_col]
    preview_feats = ', '.join(feat_cols[:12]) + ('...' if len(feat_cols) > 12 else '')

    md = (f"## ✅ Step 4 — Feature Cleaning\n\n"
          f"| | |\n|---|---|\n"
          f"| Columns before | `{df.shape[1]}` |\n"
          f"| Dropped | `{len(dropped)}` |\n"
          f"| Remaining total | `{df2.shape[1]}` |\n"
          f"| Feature columns | `{len(feat_cols)}` |\n\n"
          f"**Dropped:** `{', '.join(dropped) if dropped else 'none found'}` \n\n"
          f"**Features (first 12):** `{preview_feats}`")
    return md, gr.update(visible=True)


def step5_encoding():
    df = S.get("df")
    if df is None:
        return "⚠️  Run Step 4 first.", gr.update(visible=False)

    ds = S["dataset"]
    target_col = "category" if ds == "bot" else "type"

    # Notebook Cell 5: encode all object cols except the target
    cat_cols = [c for c in df.select_dtypes(include=["object"]).columns
                if c != target_col]

    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
    S["df"] = df

    col_list = "\n".join(f"  - `{c}`" for c in cat_cols) if cat_cols else "  _(none found)_"
    md = (f"## ✅ Step 5 — Categorical Encoding\n\n"
          f"Encoded **{len(cat_cols)}** object column(s) using `LabelEncoder`:\n\n"
          + col_list)
    return md, gr.update(visible=True)


def step6_target():
    df = S.get("df")
    if df is None:
        return "⚠️  Run Step 5 first.", gr.update(visible=False)

    ds = S["dataset"]
    target_col = "category" if ds == "bot" else "type"

    if target_col not in df.columns:
        return (f"❌ Target column `{target_col}` not found.\n\n"
                f"Columns present: `{list(df.columns)}`"), gr.update(visible=False)

    enc = LabelEncoder()
    y   = enc.fit_transform(df[target_col])

    # Drop target + any other non-feature columns still present
    cols_to_drop = [target_col]
    # BOT-IoT: 'attack' is a binary label column — not a feature
    for extra in ['attack', 'subcategory', 'label']:
        if extra in df.columns:
            cols_to_drop.append(extra)

    X = df.drop(columns=cols_to_drop)
    # Keep only numeric columns for the model
    X = X.select_dtypes(include=[np.number])

    S["target_encoder"] = enc
    S["_X_raw"] = X
    S["_y_raw"] = y

    rows = "\n".join(f"| `{c}` | `{i}` |" for i, c in enumerate(enc.classes_))
    md = (f"## ✅ Step 6 — Target Encoding\n\n"
          f"| | |\n|---|---|\n"
          f"| Feature columns (X) | `{X.shape[1]}` |\n"
          f"| Samples | `{X.shape[0]:,}` |\n"
          f"| Classes | `{len(enc.classes_)}` |\n\n"
          f"**Class mapping:**\n\n| Label | Encoded |\n|---|---|\n{rows}")
    return md, gr.update(visible=True)


def step7_scaling():
    X_raw = S.get("_X_raw")
    if X_raw is None:
        return "⚠️  Run Step 6 first.", gr.update(visible=False)

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    S["scaler"] = scaler; S["X_scaled"] = X_scaled; S["y"] = S["_y_raw"]

    md = (f"## ✅ Step 7 — Feature Scaling\n\n"
          f"| | |\n|---|---|\n"
          f"| Mean after scaling | `{X_scaled.mean():.6f}` |\n"
          f"| Std after scaling  | `{X_scaled.std():.6f}` |\n"
          f"| Shape | `{X_scaled.shape}` |")
    return md, gr.update(visible=True)


def step8_hierarchical():
    X_scaled = S.get("X_scaled")
    if X_scaled is None:
        return "⚠️  Run Step 7 first.", gr.update(visible=False)

    window_size = 5   # Fixed as per both notebooks (WINDOW_SIZE = 5)
    X_h, y_h = [], []
    for i in range(len(X_scaled) - window_size):
        X_h.append(X_scaled[i:i+window_size])
        y_h.append(S["y"][i+window_size])

    X_final = np.array(X_h, dtype=np.float32)
    y_final = np.array(y_h, dtype=np.int64)
    S["X_final"] = X_final; S["y_final"] = y_final; S["window_size"] = window_size

    md = (f"## ✅ Step 8 — Hierarchical Windows\n\n"
          f"| | |\n|---|---|\n"
          f"| Window size | `{window_size}` |\n"
          f"| Total sequences | `{X_final.shape[0]:,}` |\n"
          f"| Shape | `(samples, window, features)` = `{X_final.shape}` |")
    return md, gr.update(visible=True)


def step9_split():
    X_final = S.get("X_final")
    if X_final is None:
        return "⚠️  Run Step 8 first.", None, gr.update(visible=False)

    y_final = S["y_final"]
    X_tr,X_te,y_tr,y_te = train_test_split(X_final, y_final, test_size=0.2,
                                            random_state=42, stratify=y_final)
    dev = S["device"]
    S["X_train"]=X_tr; S["X_test"]=X_te; S["y_train"]=y_tr; S["y_test"]=y_te
    S["X_train_t"]=torch.tensor(X_tr,dtype=torch.float32).to(dev)
    S["X_test_t"] =torch.tensor(X_te,dtype=torch.float32).to(dev)
    S["y_train_t"]=torch.tensor(y_tr,dtype=torch.long).to(dev)
    S["y_test_t"] =torch.tensor(y_te,dtype=torch.long)

    enc = S["target_encoder"]; n_cls = len(enc.classes_)
    tr_c = np.bincount(y_tr, minlength=n_cls)
    te_c = np.bincount(y_te, minlength=n_cls)

    fig = light_fig(figsize=(13, 5))
    ax1 = light_ax(fig.add_subplot(121))
    ax2 = light_ax(fig.add_subplot(122))

    x = np.arange(n_cls)
    ax1.bar(x-0.18, tr_c, 0.36, label="Train", color=CYAN)
    ax1.bar(x+0.18, te_c, 0.36, label="Test",  color=PINK)
    ax1.set_xticks(x); ax1.set_xticklabels(enc.classes_, rotation=35, color="black", fontsize=8)
    ax1.set_title("Class Distribution: Train vs Test", color="black", fontsize=11)
    ax1.set_ylabel("Samples", color="black")
    ax1.legend(facecolor="white", labelcolor="black")

    ax2.pie([len(y_tr),len(y_te)], labels=["Train (80%)","Test (20%)"],
            autopct="%1.1f%%", startangle=90, colors=[CYAN,PINK],
            wedgeprops={"linewidth":1.2,"edgecolor":"#dddddd"},
            textprops={"color":"black","fontsize":10})
    ax2.set_title("Train–Test Split Ratio", color="black")
    fig.tight_layout()

    md = (f"## ✅ Step 9 — Train/Test Split\n\n"
          f"| | |\n|---|---|\n"
          f"| Train | `{len(y_tr):,}` (80%) |\n"
          f"| Test  | `{len(y_te):,}` (20%) |\n"
          f"| Strategy | Stratified |\n| Device | `{dev}` |")
    return md, fig2pil(fig), gr.update(visible=True)


def step10_weights():
    y_train = S.get("y_train")
    if y_train is None:
        return "⚠️  Run Step 9 first.", gr.update(visible=False)

    dev = S["device"]
    cw  = 1.0 / (np.bincount(y_train) + 1e-6)
    cw  = cw * (len(cw) / cw.sum())
    S["class_weights"] = torch.tensor(cw, dtype=torch.float32).to(dev)

    enc  = S["target_encoder"]
    rows = "\n".join(f"| `{c}` | {np.bincount(y_train)[i]:,} | `{cw[i]:.4f}` |"
                     for i,c in enumerate(enc.classes_))
    md = (f"## ✅ Step 10 — Class Weights\n\n"
          f"Inverse-frequency weighting for imbalanced classes.\n\n"
          f"| Class | Count | Weight |\n|---|---|---|\n{rows}")
    return md, gr.update(visible=True)


def step11_model():
    X_train = S.get("X_train")
    if X_train is None:
        return "⚠️  Run Step 9 first.", gr.update(visible=False)

    ds = S["dataset"]; dev = S["device"]
    n_cls = len(np.unique(S["y_final"])); F_in = X_train.shape[2]

    if ds == "bot":
        model = BotQKV_HAN(F_in, n_cls).to(dev)
        opt   = optim.Adam(model.parameters(), lr=1e-3)
        sch   = None
        arch  = "BiGRU(F→128) → QKVAttn(128,h=4) → BiGRU(128→128) → QKVAttn(128,h=4) → FC"
    else:
        model = TonQKV_HAN(F_in, n_cls).to(dev)
        opt   = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)
        sch   = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
        arch  = "WordEncoder(BiGRU+QKVAttn d=256,h=8) → SentenceEncoder(BiGRU+QKVAttn d=256,h=8) → FC"

    S["model"]=model; S["optimizer"]=opt; S["scheduler"]=sch

    total = sum(p.numel() for p in model.parameters())
    md = (f"## ✅ Step 11 — Model Built\n\n"
          f"**Architecture:** `{arch}`\n\n"
          f"| | |\n|---|---|\n"
          f"| Parameters | `{total:,}` |\n"
          f"| Input features | `{F_in}` |\n"
          f"| Num classes | `{n_cls}` |\n"
          f"| Device | `{dev}` |")
    return md, gr.update(visible=True)


def step12_train():
    """
    Generator — yields every 50 batches AND after every epoch for live streaming.
    With 800K samples, each epoch has ~6250 batches. Yielding every 50 batches
    means the UI refreshes ~125 times per epoch so user always sees progress.
    """
    model = S.get("model")
    if model is None:
        yield "⚠️  Run Step 11 first.", None, gr.update(visible=False)
        return

    ds = S["dataset"]
    if ds == "bot":
        epochs, batch_size = 10, 128
    else:
        epochs, batch_size = 15, 64

    dev = S["device"]
    opt = S["optimizer"]
    sch = S["scheduler"]
    cw  = S["class_weights"]
    criterion = nn.CrossEntropyLoss(weight=cw)

    X_tr = S["X_train_t"];  y_tr = S["y_train_t"]
    X_te = S["X_test_t"];   y_te = S["y_test_t"]
    N          = X_tr.size(0)
    n_batches  = (N + batch_size - 1) // batch_size
    # yield every YIELD_EVERY batches so progress is visible on large datasets
    YIELD_EVERY = max(1, n_batches // 20)   # ~20 UI refreshes per epoch

    tl_list = []; va_list = []; logs = []
    best_acc = 0.0; best_state = None

    header = (f"🚀 Training started\n"
              f"   Dataset  : {'BoT-IoT' if ds=='bot' else 'TON-IoT'}\n"
              f"   Epochs   : {epochs}\n"
              f"   Batch    : {batch_size}\n"
              f"   Samples  : {N:,}\n"
              f"   Batches/epoch: {n_batches:,}\n"
              f"{'─'*55}")
    yield header, None, gr.update(visible=False)

    for ep in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        indices    = torch.randperm(N)

        for batch_i, start in enumerate(range(0, N, batch_size)):
            batch_idx = indices[start:start + batch_size]
            xb = X_tr[batch_idx].to(dev)
            yb = y_tr[batch_idx].to(dev)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * xb.size(0)

            # ── Live batch progress update ────────────────────────────────
            if (batch_i + 1) % YIELD_EVERY == 0 or (batch_i + 1) == n_batches:
                done_pct  = (batch_i + 1) / n_batches
                filled    = int(done_pct * 25)
                bar       = "█" * filled + "░" * (25 - filled)
                avg_so_far = epoch_loss / ((batch_i + 1) * batch_size)
                ep_bar    = "█" * int(ep / epochs * 15) + "░" * (15 - int(ep / epochs * 15))
                status = (
                    header + "\n" +
                    "\n".join(logs) +
                    f"\n\nEpoch [{ep_bar}] {ep}/{epochs}\n"
                    f"  Batch [{bar}] {batch_i+1}/{n_batches}  "
                    f"({done_pct*100:.0f}%)  loss≈{avg_so_far:.4f}"
                )
                yield status, None, gr.update(visible=False)

        epoch_loss /= N
        tl_list.append(epoch_loss)

        # ── Validation after each epoch ───────────────────────────────────
        model.eval()
        with torch.no_grad():
            preds = model(X_te.to(dev)).argmax(1).cpu().numpy()
        val_acc = accuracy_score(y_te.numpy(), preds)
        va_list.append(val_acc)

        if sch:
            sch.step(val_acc)
        if val_acc > best_acc:
            best_acc   = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        lr   = opt.param_groups[0]["lr"]
        eBar = "█" * int(ep / epochs * 30) + "░" * (30 - int(ep / epochs * 30))
        line = (f"✅ Epoch {ep:>2}/{epochs}  [{eBar}]  "
                f"Loss:{epoch_loss:.4f}  ValAcc:{val_acc*100:.2f}%"
                + (f"  LR:{lr:.2e}" if ds=="ton" else ""))
        logs.append(line)
        yield header + "\n" + "\n".join(logs), None, gr.update(visible=False)

    if best_state:
        model.load_state_dict(best_state)
        logs.append(f"\n{'='*55}\n🏆 Training complete! Best Val Acc: {best_acc*100:.2f}%")

    S["train_losses"] = tl_list
    S["val_accs"]     = va_list

    # ── Build learning curve plot ────────────────────────────────────────────
    ep_range = range(1, epochs + 1)
    fig = light_fig(figsize=(13, 5))
    ax1 = light_ax(fig.add_subplot(121))
    ax2 = light_ax(fig.add_subplot(122))

    ax1.plot(ep_range, tl_list, color=CYAN, lw=2, marker="o", markersize=4)
    ax1.set_title("Training Loss", color="black", fontsize=12)
    ax1.set_xlabel("Epoch", color="black"); ax1.set_ylabel("CrossEntropy Loss", color="black")
    ax1.grid(alpha=0.2, color=BORDER)

    ax2.plot(ep_range, [v*100 for v in va_list], color=GREEN, lw=2, marker="o", markersize=4)
    ax2.set_title("Validation Accuracy", color="black", fontsize=12)
    ax2.set_xlabel("Epoch", color="black"); ax2.set_ylabel("Accuracy (%)", color="black")
    ax2.grid(alpha=0.2, color=BORDER)

    fig.tight_layout()

    # Final yield with plot and reveal next step
    yield "\n".join(logs), fig2pil(fig), gr.update(visible=True)


def step13_loss_viz():
    tl=S.get("train_losses")
    if not tl:
        return "⚠️  Run Step 12 first.", None, gr.update(visible=False)

    va=S["val_accs"]; ep=range(1,len(tl)+1)
    fig=light_fig(figsize=(15,5))
    ax1=light_ax(fig.add_subplot(131))
    ax2=light_ax(fig.add_subplot(132))
    ax3=light_ax(fig.add_subplot(133))

    ax1.plot(ep,tl,color=BLUE,lw=2)
    ax1.set_title("Training Loss",color="black"); ax1.set_xlabel("Epoch",color="black")
    ax1.set_ylabel("Loss",color="black"); ax1.grid(alpha=0.2,color=BORDER)

    ax2.plot(ep,[v*100 for v in va],color=GREEN,lw=2)
    ax2.set_title("Validation Accuracy",color="black"); ax2.set_xlabel("Epoch",color="black")
    ax2.set_ylabel("Accuracy (%)",color="black"); ax2.grid(alpha=0.2,color=BORDER)

    imp=[0]+[tl[i-1]-tl[i] for i in range(1,len(tl))]
    ax3.bar(ep,imp,color=[GREEN if v>=0 else PINK for v in imp])
    ax3.axhline(0,color=BORDER,lw=0.8)
    ax3.set_title("Loss Δ per Epoch",color="black"); ax3.set_xlabel("Epoch",color="black")
    ax3.set_ylabel("Δ Loss",color="black"); ax3.grid(alpha=0.2,color=BORDER)
    fig.tight_layout()

    md=(f"## ✅ Step 13 — Training Curves\n\n"
        f"| | |\n|---|---|\n"
        f"| Initial loss | `{tl[0]:.4f}` |\n"
        f"| Final loss   | `{tl[-1]:.4f}` |\n"
        f"| Best val acc | `{max(va)*100:.2f}%` |")
    return md, fig2pil(fig), gr.update(visible=True)


def step14_evaluate():
    model=S.get("model")
    if model is None:
        return "⚠️  Run Step 12 first.", gr.update(visible=False)

    X_te=S["X_test_t"]; y_te=S["y_test_t"]
    enc=S["target_encoder"]; dev=S["device"]; bs=256
    model.eval(); y_true=[]; y_pred=[]
    with torch.no_grad():
        for start in range(0,X_te.size(0),bs):
            xb=X_te[start:start+bs].to(dev)
            y_pred.extend(model(xb).argmax(1).cpu().tolist())
            y_true.extend(y_te[start:start+bs].tolist())

    S["y_true"]=y_true; S["y_pred"]=y_pred
    acc=accuracy_score(y_true,y_pred)
    prec,rec,f1,_=precision_recall_fscore_support(y_true,y_pred,average="weighted",zero_division=0)
    rep=classification_report(y_true,y_pred,target_names=enc.classes_,zero_division=0)

    md=(f"## ✅ Step 14 — Final Evaluation\n\n"
        f"| Metric | Score |\n|---|---|\n"
        f"| **Accuracy**  | `{acc*100:.2f}%` |\n"
        f"| **Precision** | `{prec*100:.2f}%` |\n"
        f"| **Recall**    | `{rec*100:.2f}%` |\n"
        f"| **F1-Score**  | `{f1*100:.2f}%` |\n\n"
        f"```\n{rep}\n```")
    return md, gr.update(visible=True)


def step15_confusion():
    y_true=S.get("y_true")
    if y_true is None:
        return "⚠️  Run Step 14 first.", None, None, gr.update(visible=False)

    y_pred=S["y_pred"]; enc=S["target_encoder"]; ds=S["dataset"]
    cm=confusion_matrix(y_true,y_pred)
    acc=accuracy_score(y_true,y_pred)
    prec,rec,f1,_=precision_recall_fscore_support(y_true,y_pred,average="weighted",zero_division=0)
    label=f"{'BoT-IoT' if ds=='bot' else 'TON-IoT'} QKV-HAN"

    # Confusion matrix
    fig1=light_fig(figsize=(10,8))
    ax=light_ax(fig1.add_subplot(111))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
                xticklabels=enc.classes_,yticklabels=enc.classes_,
                linewidths=0.5,linecolor=DARK,ax=ax,cbar_kws={"shrink":0.8})
    ax.set_title(f"Confusion Matrix – {label}",color="black",fontsize=13,pad=12)
    ax.set_xlabel("Predicted Label",color="black"); ax.set_ylabel("True Label",color="black")
    ax.tick_params(colors="black",labelsize=8)
    plt.setp(ax.get_xticklabels(),rotation=45,ha="right",color="black")
    plt.setp(ax.get_yticklabels(),rotation=0,color="black")
    fig1.tight_layout()

    # 6-panel performance charts
    metrics={"Accuracy":acc*100,"Precision":prec*100,"Recall":rec*100,"F1-score":f1*100}
    names=list(metrics.keys()); values=list(metrics.values())

    fig2=light_fig(figsize=(16,10))
    axs=[light_ax(fig2.add_subplot(2,3,i+1)) for i in range(6)]

    bars=axs[0].bar(names,values,color=PAL[:4],edgecolor="#dddddd")
    for b in bars:
        axs[0].text(b.get_x()+b.get_width()/2,b.get_height(),
                    f"{b.get_height():.2f}%",ha="center",va="bottom",color="black",fontsize=9)
    axs[0].set_ylim(min(values)-1,101); axs[0].set_title("Bar Chart",color="black")
    axs[0].set_ylabel("Score (%)",color="black"); axs[0].grid(axis="y",alpha=0.25,color=BORDER)

    axs[1].pie(values,labels=names,autopct="%1.2f%%",colors=PAL[:4],
               wedgeprops={"linewidth":1,"edgecolor":"#dddddd"},
               textprops={"color":"black","fontsize":8})
    axs[1].set_title("Pie Chart",color="black")

    axs[2].plot(names,values,marker="o",lw=2,color=CYAN)
    for i,v in enumerate(values):
        axs[2].text(i,v+0.1,f"{v:.2f}%",ha="center",color="black",fontsize=8)
    axs[2].set_ylim(min(values)-1,101); axs[2].set_title("Line Chart",color="black")
    axs[2].set_ylabel("Score (%)",color="black"); axs[2].grid(alpha=0.25,color=BORDER)
    axs[2].tick_params(axis="x",colors="white"); axs[2].tick_params(axis="y",colors="white")

    axs[3].hist(values,bins=4,color=BLUE,edgecolor="#dddddd")
    axs[3].set_title("Histogram",color="black")
    axs[3].set_xlabel("Score (%)",color="black"); axs[3].set_ylabel("Frequency",color="black")
    axs[3].grid(alpha=0.25,color=BORDER)

    axs[4].boxplot(values,vert=True,patch_artist=True,
                   boxprops={"facecolor":CYAN,"color":"white"},
                   medianprops={"color":PINK,"lw":2},
                   whiskerprops={"color":"white"},capprops={"color":"white"})
    axs[4].set_title("Box Plot",color="black")
    axs[4].set_ylabel("Score (%)",color="black"); axs[4].grid(alpha=0.25,color=BORDER)

    vp=axs[5].violinplot(values,showmeans=True)
    for pc in vp["bodies"]: pc.set_facecolor(PINK); pc.set_alpha(0.7)
    axs[5].set_title("Violin Plot",color="black")
    axs[5].set_ylabel("Score (%)",color="black"); axs[5].grid(alpha=0.25,color=BORDER)
    fig2.tight_layout()

    md="## ✅ Step 15 — Confusion Matrix & Performance Charts"
    return md, fig2pil(fig1), fig2pil(fig2), gr.update(visible=True)


def step16_comparison():
    y_true=S.get("y_true")
    if y_true is None:
        return "⚠️  Run Step 14 first.", None, None

    y_pred=S["y_pred"]; enc=S["target_encoder"]; ds=S["dataset"]
    from sklearn.metrics import precision_recall_fscore_support as prfs
    precs,recs,f1s,_=prfs(y_true,y_pred,average=None,zero_division=0)
    classes=enc.classes_; x=np.arange(len(classes))

    # Attack-wise bar
    fig1=light_fig(figsize=(14,6))
    ax=light_ax(fig1.add_subplot(111))
    ax.bar(x-0.25,precs,0.25,label="Precision",color=CYAN)
    ax.bar(x,     recs, 0.25,label="Recall",   color=GREEN)
    ax.bar(x+0.25,f1s,  0.25,label="F1-score", color=PINK)
    for i in range(len(classes)):
        ax.text(x[i]-0.25,precs[i]+0.02,f"{precs[i]:.2f}",ha="center",color="black",fontsize=8)
        ax.text(x[i],     recs[i]+0.02, f"{recs[i]:.2f}", ha="center",color="black",fontsize=8)
        ax.text(x[i]+0.25,f1s[i]+0.02,  f"{f1s[i]:.2f}",  ha="center",color="black",fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(classes,rotation=35,color="black",fontsize=9)
    ax.set_ylim(0,1.12)
    ax.set_title(f"Attack-wise Performance — {'BoT-IoT' if ds=='bot' else 'TON-IoT'}",
                 color="black",fontsize=12)
    ax.set_ylabel("Score",color="black"); ax.legend(facecolor=CARD,labelcolor="black")
    ax.grid(axis="y",alpha=0.25,color=BORDER)
    fig1.tight_layout()

    # TON vs BOT comparison (reference values from notebooks)
    m_labels=["Accuracy","Precision","Recall","F1-score"]
    ton_v=[98.92,99.01,98.92,98.95]; bot_v=[99.99,99.98,99.99,99.98]
    xc=np.arange(4)

    fig2=light_fig(figsize=(16,10))
    fig2.suptitle("TON-IoT vs BoT-IoT — Full Comparison",color="black",fontsize=14)
    caxs=[light_ax(fig2.add_subplot(2,2,i+1)) for i in range(4)]

    caxs[0].bar(xc-0.18,ton_v,0.36,label="TON-IoT",color=CYAN)
    caxs[0].bar(xc+0.18,bot_v,0.36,label="BoT-IoT",color=PINK)
    for i in range(4):
        caxs[0].text(xc[i]-0.18,ton_v[i]+0.04,f"{ton_v[i]:.2f}%",ha="center",color="black",fontsize=8)
        caxs[0].text(xc[i]+0.18,bot_v[i]+0.04,f"{bot_v[i]:.2f}%",ha="center",color="black",fontsize=8)
    caxs[0].set_xticks(xc); caxs[0].set_xticklabels(m_labels,color="black",fontsize=9)
    caxs[0].set_ylim(98,100.3); caxs[0].set_title("Overall Metrics (Bar)",color="black")
    caxs[0].set_ylabel("Score (%)",color="black"); caxs[0].legend(facecolor=CARD,labelcolor="black")
    caxs[0].grid(axis="y",alpha=0.25,color=BORDER)

    caxs[1].plot(m_labels,ton_v,marker="o",lw=2,label="TON-IoT",color=CYAN)
    caxs[1].plot(m_labels,bot_v,marker="s",lw=2,label="BoT-IoT",color=PINK)
    caxs[1].set_ylim(98,100.3); caxs[1].set_title("Overall Metrics (Line)",color="black")
    caxs[1].set_ylabel("Score (%)",color="black"); caxs[1].legend(facecolor=CARD,labelcolor="black")
    caxs[1].grid(alpha=0.25,color=BORDER)
    caxs[1].tick_params(axis="x",colors="white"); caxs[1].tick_params(axis="y",colors="white")

    avg_l=["Macro Avg F1","Weighted Avg F1"]
    ton_a=[97.0,99.0]; bot_a=[87.0,100.0]; xa=np.arange(2)
    caxs[2].bar(xa-0.18,ton_a,0.36,label="TON-IoT",color=CYAN)
    caxs[2].bar(xa+0.18,bot_a,0.36,label="BoT-IoT",color=PINK)
    caxs[2].set_xticks(xa); caxs[2].set_xticklabels(avg_l,color="black",fontsize=9)
    caxs[2].set_ylim(80,102); caxs[2].set_title("Macro vs Weighted F1",color="black")
    caxs[2].set_ylabel("F1-score (%)",color="black"); caxs[2].legend(facecolor=CARD,labelcolor="black")
    caxs[2].grid(axis="y",alpha=0.25,color=BORDER)

    atk_l=["High-freq Attacks","Rare Attacks"]
    ton_atk=[0.99,0.77]; bot_atk=[1.00,0.50]; xat=np.arange(2)
    caxs[3].bar(xat-0.18,ton_atk,0.36,label="TON-IoT",color=CYAN)
    caxs[3].bar(xat+0.18,bot_atk,0.36,label="BoT-IoT",color=PINK)
    caxs[3].set_xticks(xat); caxs[3].set_xticklabels(atk_l,color="black",fontsize=9)
    caxs[3].set_ylim(0.4,1.05); caxs[3].set_title("Attack-Type F1 Sensitivity",color="black")
    caxs[3].set_ylabel("F1-score",color="black"); caxs[3].legend(facecolor=CARD,labelcolor="black")
    caxs[3].grid(axis="y",alpha=0.25,color=BORDER)
    fig2.tight_layout()

    md="## ✅ Step 16 — Attack-wise & TON-IoT vs BoT-IoT Comparison\n\nPipeline complete! 🎉"
    return md, fig2pil(fig1), fig2pil(fig2)


# =============================================================================
#  GRADIO UI  — Clean Modern Professional
# =============================================================================

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@300;400;500;600;700&family=DM+Mono:wght@400;500&display=swap');

:root {
  --bg:        #f8fafc;
  --bg2:       #f1f5f9;
  --white:     #ffffff;
  --border:    #e2e8f0;
  --border2:   #cbd5e1;
  --text:      #0f172a;
  --text2:     #334155;
  --muted:     #64748b;
  --blue:      #3b82f6;
  --blue-d:    #2563eb;
  --blue-l:    #eff6ff;
  --blue-b:    #bfdbfe;
  --indigo:    #6366f1;
  --green:     #10b981;
  --green-l:   #ecfdf5;
  --orange:    #f97316;
  --orange-l:  #fff7ed;
  --red:       #ef4444;
  --shadow-sm: 0 1px 3px rgba(0,0,0,0.06), 0 1px 2px rgba(0,0,0,0.04);
  --shadow:    0 4px 6px -1px rgba(0,0,0,0.07), 0 2px 4px -1px rgba(0,0,0,0.04);
  --shadow-md: 0 10px 15px -3px rgba(0,0,0,0.08), 0 4px 6px -2px rgba(0,0,0,0.04);
  --radius:    12px;
  --radius-sm: 8px;
}

*, *::before, *::after { box-sizing: border-box; }

body, .gradio-container, .main, .wrap {
  background: var(--bg) !important;
  font-family: 'Plus Jakarta Sans', -apple-system, sans-serif !important;
  color: var(--text) !important;
}

/* ── Cards / Groups ── */
.block, .gr-group, .gr-box {
  background: var(--white) !important;
  border: 1px solid var(--border) !important;
  border-radius: var(--radius) !important;
  box-shadow: var(--shadow-sm) !important;
  padding: 24px 28px !important;
  margin-bottom: 16px !important;
  transition: box-shadow 0.2s ease, border-color 0.2s ease;
}
.block:hover {
  box-shadow: var(--shadow) !important;
  border-color: var(--border2) !important;
}

/* ── Typography ── */
h1, h2, h3, h4 { color: var(--text) !important; font-family: 'Plus Jakarta Sans', sans-serif !important; }
.prose, p, li { color: var(--text2) !important; font-size: 0.875rem !important; line-height: 1.6 !important; }
label, .label-wrap span { color: var(--muted) !important; font-size: 0.78rem !important; font-weight: 500 !important; letter-spacing: 0.02em !important; }

/* ── Buttons ── */
button { font-family: 'Plus Jakarta Sans', sans-serif !important; font-weight: 600 !important; font-size: 0.875rem !important; transition: all 0.15s ease !important; }

button.primary, [data-testid="primary"] {
  background: linear-gradient(135deg, #3b82f6, #6366f1) !important;
  border: none !important;
  color: white !important;
  border-radius: 8px !important;
  padding: 11px 28px !important;
  box-shadow: 0 4px 14px rgba(99,102,241,0.35) !important;
  letter-spacing: 0.01em !important;
}
button.primary:hover {
  background: linear-gradient(135deg, #2563eb, #4f46e5) !important;
  box-shadow: 0 6px 20px rgba(99,102,241,0.45) !important;
  transform: translateY(-1px) !important;
}
button.primary:active { transform: translateY(0) !important; }

button.secondary {
  background: var(--white) !important;
  border: 1.5px solid var(--border2) !important;
  color: var(--text2) !important;
  border-radius: 8px !important;
}
button.secondary:hover {
  border-color: var(--blue) !important;
  color: var(--blue) !important;
  background: var(--blue-l) !important;
}

/* ── Inputs ── */
input, textarea, select, .gr-textbox input, .gr-textbox textarea {
  background: var(--bg) !important;
  border: 1.5px solid var(--border) !important;
  border-radius: var(--radius-sm) !important;
  color: var(--text) !important;
  font-family: 'Plus Jakarta Sans', sans-serif !important;
  font-size: 0.875rem !important;
  padding: 10px 14px !important;
  transition: border-color 0.15s, box-shadow 0.15s;
}
input:focus, textarea:focus {
  border-color: var(--blue) !important;
  box-shadow: 0 0 0 3px rgba(59,130,246,0.12) !important;
  outline: none !important;
  background: var(--white) !important;
}

/* Training log */
.gr-textbox textarea {
  background: #0f172a !important;
  color: #a3e635 !important;
  border-color: #1e293b !important;
  font-family: 'DM Mono', monospace !important;
  font-size: 0.8rem !important;
  line-height: 1.7 !important;
}

/* ── Radio ── */
.gr-radio label {
  background: var(--white) !important;
  border: 1.5px solid var(--border) !important;
  border-radius: var(--radius-sm) !important;
  padding: 14px 20px !important;
  cursor: pointer;
  transition: all 0.15s;
  color: var(--text2) !important;
  font-size: 0.875rem !important;
  font-weight: 500 !important;
  text-transform: none !important;
  letter-spacing: normal !important;
}
.gr-radio label:hover {
  border-color: var(--blue) !important;
  background: var(--blue-l) !important;
  color: var(--blue-d) !important;
}
.gr-radio input[type=radio]:checked + label {
  background: var(--blue-l) !important;
  border-color: var(--blue) !important;
  color: var(--blue-d) !important;
  box-shadow: 0 0 0 3px rgba(59,130,246,0.12) !important;
}

/* ── File upload ── */
.gr-file, [data-testid="file"], .upload-container {
  background: var(--bg) !important;
  border: 2px dashed var(--border2) !important;
  border-radius: var(--radius) !important;
  transition: all 0.2s;
  min-height: 100px !important;
}
.gr-file:hover, .upload-container:hover {
  border-color: var(--blue) !important;
  background: var(--blue-l) !important;
}

/* ── Dataframe ── */
.gr-dataframe table { background: var(--white) !important; font-size: 0.8rem !important; border-radius: var(--radius-sm) !important; overflow: hidden; }
.gr-dataframe th { background: var(--bg2) !important; color: var(--muted) !important; font-weight: 600 !important; font-size: 0.72rem !important; letter-spacing: 0.05em !important; text-transform: uppercase !important; padding: 10px 14px !important; border-bottom: 1px solid var(--border) !important; }
.gr-dataframe td { color: var(--text2) !important; padding: 9px 14px !important; border-bottom: 1px solid var(--border) !important; }
.gr-dataframe tr:hover td { background: var(--bg) !important; }

/* ── Images ── */
.gr-image, [data-testid="image"] { background: var(--bg) !important; border: 1px solid var(--border) !important; border-radius: var(--radius) !important; overflow: hidden; }

/* ── Markdown ── */
.prose table { background: var(--white) !important; border: 1px solid var(--border) !important; border-radius: var(--radius-sm) !important; overflow: hidden; font-size: 0.84rem !important; }
.prose table th { background: var(--bg2) !important; color: var(--muted) !important; font-weight: 600 !important; font-size: 0.72rem !important; text-transform: uppercase !important; letter-spacing: 0.05em !important; padding: 9px 14px !important; border-bottom: 1px solid var(--border) !important; }
.prose table td { color: var(--text2) !important; padding: 8px 14px !important; border-bottom: 1px solid var(--border) !important; }
.prose table tr:last-child td { border-bottom: none !important; }
.prose code { background: var(--bg2) !important; color: var(--indigo) !important; border: 1px solid var(--border) !important; border-radius: 5px !important; padding: 1px 6px !important; font-family: 'DM Mono', monospace !important; font-size: 0.82em !important; }
.prose blockquote { border-left: 3px solid var(--orange) !important; background: var(--orange-l) !important; padding: 10px 16px !important; margin: 12px 0 !important; border-radius: 0 8px 8px 0 !important; color: var(--text2) !important; }
.prose strong { color: var(--text) !important; }

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 5px; height: 5px; }
::-webkit-scrollbar-track { background: var(--bg); }
::-webkit-scrollbar-thumb { background: var(--border2); border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: var(--blue); }

/* ── Step card header ── */
.step-card-header { display:flex; align-items:center; gap:14px; padding-bottom:18px; margin-bottom:18px; border-bottom:1px solid var(--border); }
.step-num { width:36px; height:36px; border-radius:10px; display:flex; align-items:center; justify-content:center; font-size:0.78rem; font-weight:700; flex-shrink:0; }

/* Phase colours */
.phase-data   { background:#eff6ff; color:#2563eb; }
.phase-proc   { background:#fef3c7; color:#d97706; }
.phase-train  { background:#f5f3ff; color:#7c3aed; }
.phase-eval   { background:#ecfdf5; color:#059669; }

/* ── Connector ── */
.step-connector { display:flex; justify-content:center; height:20px; margin:0; }

/* ── Badge ── */
.badge { display:inline-flex; align-items:center; gap:5px; padding:3px 10px; border-radius:20px; font-size:0.7rem; font-weight:600; letter-spacing:0.03em; }
.badge-blue  { background:var(--blue-l); color:var(--blue-d); }
.badge-green { background:var(--green-l); color:#059669; }

/* ── Fade in ── */
@keyframes fadeUp { from { opacity:0; transform:translateY(8px); } to { opacity:1; transform:translateY(0); } }
.gr-group { animation: fadeUp 0.3s ease forwards; }
"""

# ── Step card header builder ─────────────────────────────────────────────────
def step_html(num, icon, title, subtitle, phase="data"):
    phase_class = {"data":"phase-data","process":"phase-proc","train":"phase-train","eval":"phase-eval"}.get(phase,"phase-data")
    phase_labels = {"data":"Data Prep","process":"Processing","train":"Training","eval":"Evaluation"}
    phase_label  = phase_labels.get(phase,"Step")
    badge_color  = {"data":"badge-blue","process":"badge-blue","train":"badge-blue","eval":"badge-green"}.get(phase,"badge-blue")
    return f"""<div style="display:flex;align-items:center;gap:14px;padding-bottom:18px;margin-bottom:18px;border-bottom:1px solid #e2e8f0;">
  <div style="width:40px;height:40px;border-radius:10px;display:flex;align-items:center;justify-content:center;font-size:1.1rem;flex-shrink:0;" class="{phase_class}">{icon}</div>
  <div style="flex:1;">
    <div style="display:flex;align-items:center;gap:8px;margin-bottom:3px;">
      <span style="font-size:0.95rem;font-weight:700;color:#0f172a;font-family:'Plus Jakarta Sans',sans-serif;">{title}</span>
      <span style="display:inline-flex;align-items:center;padding:2px 9px;border-radius:20px;font-size:0.68rem;font-weight:600;" class="{badge_color}">STEP {num}</span>
    </div>
    <div style="font-size:0.78rem;color:#64748b;font-family:'Plus Jakarta Sans',sans-serif;">{subtitle}</div>
  </div>
</div>"""

DIVIDER = """<div style="display:flex;justify-content:center;align-items:center;height:16px;margin:0;">
  <div style="width:1px;height:16px;background:linear-gradient(180deg,#e2e8f0,transparent);"></div>
</div>"""

HEADER_HTML = """
<div style="
  background:white;
  border:1px solid #e2e8f0;
  border-radius:16px;
  padding:36px 40px;
  margin-bottom:8px;
  box-shadow:0 1px 3px rgba(0,0,0,0.06);
  position:relative;
  overflow:hidden;
">
  <!-- Subtle gradient accent top -->
  <div style="position:absolute;top:0;left:0;right:0;height:3px;
    background:linear-gradient(90deg,#3b82f6,#6366f1,#8b5cf6);"></div>

  <div style="display:flex;align-items:flex-start;gap:20px;flex-wrap:wrap;">
    <!-- Left: title block -->
    <div style="flex:1;min-width:280px;">
      <div style="display:flex;align-items:center;gap:12px;margin-bottom:8px;">
        <div style="width:48px;height:48px;border-radius:12px;
          background:linear-gradient(135deg,#eff6ff,#e0e7ff);
          display:flex;align-items:center;justify-content:center;font-size:1.5rem;
          box-shadow:0 2px 8px rgba(99,102,241,0.15);">🛡️</div>
        <div>
          <h1 style="margin:0;font-size:1.4rem;font-weight:800;color:#0f172a;
            font-family:'Plus Jakarta Sans',sans-serif;letter-spacing:-0.02em;line-height:1.2;">
            HAN-QKV Cyber Attack Detection
          </h1>
          <p style="margin:4px 0 0 0;font-size:0.8rem;color:#64748b;
            font-family:'Plus Jakarta Sans',sans-serif;">
            Hierarchical Attention Network with QKV Attention
          </p>
        </div>
      </div>
    </div>

    <!-- Right: stat pills -->
    <div style="display:flex;gap:10px;flex-wrap:wrap;align-items:center;">
      <div style="background:#eff6ff;border:1px solid #bfdbfe;border-radius:10px;padding:10px 16px;text-align:center;">
        <div style="font-size:0.65rem;font-weight:600;color:#93c5fd;text-transform:uppercase;letter-spacing:0.08em;margin-bottom:2px;">BoT-IoT</div>
        <div style="font-size:0.85rem;font-weight:700;color:#2563eb;">99.99% F1</div>
      </div>
      <div style="background:#f5f3ff;border:1px solid #ddd6fe;border-radius:10px;padding:10px 16px;text-align:center;">
        <div style="font-size:0.65rem;font-weight:600;color:#c4b5fd;text-transform:uppercase;letter-spacing:0.08em;margin-bottom:2px;">TON-IoT</div>
        <div style="font-size:0.85rem;font-weight:700;color:#7c3aed;">98.92% F1</div>
      </div>
      <div style="background:#ecfdf5;border:1px solid #a7f3d0;border-radius:10px;padding:10px 16px;text-align:center;">
        <div style="font-size:0.65rem;font-weight:600;color:#6ee7b7;text-transform:uppercase;letter-spacing:0.08em;margin-bottom:2px;">Pipeline</div>
        <div style="font-size:0.85rem;font-weight:700;color:#059669;">16 Steps</div>
      </div>
    </div>
  </div>
</div>
"""

FOOTER_HTML = """<div style="
  text-align:center;padding:20px 0 8px 0;margin-top:32px;
  border-top:1px solid #e2e8f0;
  font-size:0.75rem;color:#94a3b8;
  font-family:'Plus Jakarta Sans',sans-serif;
">
  HAN-QKV · Hierarchical Attention Network with QKV Attention · BoT-IoT & TON-IoT Cyber Attack Detection
</div>"""

with gr.Blocks(css=CSS, title="HAN-QKV Cyber Attack Detection") as demo:

    gr.HTML(HEADER_HTML)

    # ── Dataset selector ─────────────────────────────────────────────────────
    with gr.Group():
        gr.HTML("""<div style="display:flex;align-items:center;gap:10px;padding-bottom:16px;margin-bottom:16px;border-bottom:1px solid #e2e8f0;">
          <div style="width:8px;height:8px;border-radius:50%;background:#3b82f6;"></div>
          <span style="font-size:0.9rem;font-weight:700;color:#0f172a;font-family:'Plus Jakarta Sans',sans-serif;">Select Dataset</span>
          <span style="margin-left:auto;font-size:0.72rem;color:#94a3b8;">Choose your IoT dataset to begin</span>
        </div>""")
        dataset_radio = gr.Radio(
            choices=["📦 BoT-IoT Dataset", "🌐 TON-IoT Dataset"],
            value="📦 BoT-IoT Dataset", label="",
            info="BoT-IoT → reduced_data_4.csv (5 attack classes)  ·  TON-IoT → train_test_network.csv"
        )

    #gr.HTML(DIVIDER)

    # ── Step 1 ────────────────────────────────────────────────────────────────
    with gr.Group():
        gr.HTML(step_html(1, "⚙️", "Imports & Seed", "Initialize all libraries and set random seed 42 for reproducibility", "data"))
        btn1 = gr.Button("Run Step 1 — Initialize", variant="primary")
        out1 = gr.Markdown()

    #gr.HTML(DIVIDER)

    # ── Step 2 ────────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp2:
        gr.HTML(step_html(2, "📂", "Data Loading", "Upload your CSV dataset file", "data"))
        gr.HTML("""<div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:12px 16px;margin-bottom:14px;display:flex;gap:20px;">
          <span style="font-size:0.8rem;color:#475569;"><strong style="color:#2563eb;">BoT-IoT</strong> → reduced_data_4.csv</span>
          <span style="font-size:0.8rem;color:#475569;"><strong style="color:#7c3aed;">TON-IoT</strong> → train_test_network.csv</span>
        </div>""")
        file2 = gr.File(label="Upload CSV file", file_types=[".csv"], file_count="single")
        btn2  = gr.Button("Run Step 2 — Load Dataset", variant="primary")
        out2  = gr.Markdown()
        prev2 = gr.Dataframe(label="Dataset Preview — first 5 rows", interactive=False, visible=False)

    #gr.HTML(DIVIDER)

    # ── Step 3 ────────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp3:
        gr.HTML(step_html(3, "📊", "Target Distribution", "Visualize attack class counts and proportions", "data"))
        btn3 = gr.Button("Run Step 3 — Analyze Distribution", variant="primary")
        out3 = gr.Markdown()
        img3 = gr.Image(label="Distribution Charts", type="pil")

    #gr.HTML(DIVIDER)

    # ── Step 4 ────────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp4:
        gr.HTML(step_html(4, "🧹", "Feature Cleaning", "Drop irrelevant columns — IP addresses, timestamps, IDs", "process"))
        btn4 = gr.Button("Run Step 4 — Clean Features", variant="primary")
        out4 = gr.Markdown()

    #gr.HTML(DIVIDER)

    # ── Step 5 ────────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp5:
        gr.HTML(step_html(5, "🔤", "Categorical Encoding", "Apply LabelEncoder to all object-type columns", "process"))
        btn5 = gr.Button("Run Step 5 — Encode Categories", variant="primary")
        out5 = gr.Markdown()

    #gr.HTML(DIVIDER)

    # ── Step 6 ────────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp6:
        gr.HTML(step_html(6, "🎯", "Target Encoding", "Encode target labels and split features X from target y", "process"))
        btn6 = gr.Button("Run Step 6 — Encode Target", variant="primary")
        out6 = gr.Markdown()

    #gr.HTML(DIVIDER)

    # ── Step 7 ────────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp7:
        gr.HTML(step_html(7, "📐", "Feature Scaling", "StandardScaler normalization and correlation heatmap", "process"))
        btn7 = gr.Button("Run Step 7 — Scale Features", variant="primary")
        out7 = gr.Markdown()


    #gr.HTML(DIVIDER)

    # ── Step 8 ────────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp8:
        gr.HTML(step_html(8, "🪟", "Hierarchical Windows", "Create sliding window sequences for hierarchical input", "process"))
        gr.HTML("""<div style="display:inline-flex;align-items:center;gap:6px;background:#fff7ed;border:1px solid #fed7aa;border-radius:6px;padding:6px 12px;margin-bottom:12px;font-size:0.78rem;color:#92400e;">
          📌 Window size fixed to <strong style="color:#d97706;">5</strong> — matches both notebooks exactly
        </div>""")
        btn8 = gr.Button("Run Step 8 — Create Windows", variant="primary")
        out8 = gr.Markdown()


    #gr.HTML(DIVIDER)

    # ── Step 9 ────────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp9:
        gr.HTML(step_html(9, "✂️", "Train / Test Split", "Stratified 80/20 split with class balance preserved", "process"))
        btn9 = gr.Button("Run Step 9 — Split Dataset", variant="primary")
        out9 = gr.Markdown()
        img9 = gr.Image(label="Split Distribution", type="pil")

    #gr.HTML(DIVIDER)

    # ── Step 10 ───────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp10:
        gr.HTML(step_html(10, "⚖️", "Class Weights", "Compute inverse-frequency weights for class imbalance", "process"))
        btn10 = gr.Button("Run Step 10 — Compute Weights", variant="primary")
        out10 = gr.Markdown()

    #gr.HTML(DIVIDER)

    # ── Step 11 ───────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp11:
        gr.HTML(step_html(11, "🧠", "Model Architecture", "Build QKV-HAN — BiGRU with Multi-Head QKV Attention", "train"))
        btn11 = gr.Button("Run Step 11 — Build Model", variant="primary")
        out11 = gr.Markdown()

    #gr.HTML(DIVIDER)

    # ── Step 12 ───────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp12:
        gr.HTML(step_html(12, "🚀", "Training Loop", "CrossEntropyLoss + Adam optimizer with live batch progress", "train"))
        gr.HTML("""<div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:16px;">
          <div style="background:#eff6ff;border:1px solid #bfdbfe;border-radius:8px;padding:12px 16px;">
            <div style="font-size:0.68rem;font-weight:700;color:#93c5fd;text-transform:uppercase;letter-spacing:0.08em;margin-bottom:4px;">BoT-IoT</div>
            <div style="font-size:0.82rem;color:#1e40af;font-family:'DM Mono',monospace;">epochs=10 · batch=128 · lr=1e-3</div>
          </div>
          <div style="background:#f5f3ff;border:1px solid #ddd6fe;border-radius:8px;padding:12px 16px;">
            <div style="font-size:0.68rem;font-weight:700;color:#c4b5fd;text-transform:uppercase;letter-spacing:0.08em;margin-bottom:4px;">TON-IoT</div>
            <div style="font-size:0.82rem;color:#5b21b6;font-family:'DM Mono',monospace;">epochs=15 · batch=64 · lr=3e-4</div>
          </div>
        </div>""")
        btn12 = gr.Button("🚀  Run Step 12 — Start Training", variant="primary")
        out12 = gr.Textbox(label="Live Training Log", lines=14, max_lines=22)
        img12 = gr.Image(label="Learning Curves", type="pil")

    #gr.HTML(DIVIDER)

    # ── Step 13 ───────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp13:
        gr.HTML(step_html(13, "📉", "Loss Visualisation", "3-panel analysis: training loss, validation accuracy, loss delta", "eval"))
        btn13 = gr.Button("Run Step 13 — Plot Curves", variant="primary")
        out13 = gr.Markdown()
        img13 = gr.Image(label="Training Curves — 3 Panel", type="pil")

    #gr.HTML(DIVIDER)

    # ── Step 14 ───────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp14:
        gr.HTML(step_html(14, "📈", "Final Evaluation", "Accuracy, Precision, Recall, F1-score with full classification report", "eval"))
        btn14 = gr.Button("Run Step 14 — Evaluate Model", variant="primary")
        out14 = gr.Markdown()

    #gr.HTML(DIVIDER)

    # ── Step 15 ───────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp15:
        gr.HTML(step_html(15, "🗂️", "Confusion Matrix & Charts", "Confusion matrix heatmap plus 6-panel performance visualization", "eval"))
        btn15 = gr.Button("Run Step 15 — Generate Charts", variant="primary")
        out15 = gr.Markdown()
        with gr.Row():
            img15a = gr.Image(label="Confusion Matrix", type="pil")
            img15b = gr.Image(label="6-Panel Performance Charts", type="pil")

    #gr.HTML(DIVIDER)

    # ── Step 16 ───────────────────────────────────────────────────────────────
    with gr.Group(visible=False) as grp16:
        gr.HTML(step_html(16, "🔥", "Attack Comparison", "Attack-wise P/R/F1 bars and TON-IoT vs BoT-IoT side-by-side", "eval"))
        btn16 = gr.Button("Run Step 16 — Compare Datasets", variant="primary")
        out16 = gr.Markdown()
        with gr.Row():
            img16a = gr.Image(label="Attack-wise Performance", type="pil")
            img16b = gr.Image(label="TON-IoT vs BoT-IoT Comparison", type="pil")

    gr.HTML(FOOTER_HTML)

    # ── Wire up — zero backend changes ────────────────────────────────────────
    btn1.click(step1_seeds,        inputs=dataset_radio,  outputs=[out1, grp2])
    btn2.click(step2_load,         inputs=file2,           outputs=[out2, prev2, grp3])
    btn3.click(step3_distribution,                         outputs=[out3, img3, grp4])
    btn4.click(step4_cleaning,                             outputs=[out4, grp5])
    btn5.click(step5_encoding,                             outputs=[out5, grp6])
    btn6.click(step6_target,                               outputs=[out6, grp7])
    btn7.click(step7_scaling,                              outputs=[out7, grp8])
    btn8.click(step8_hierarchical,                         outputs=[out8, grp9])
    btn9.click(step9_split,                                outputs=[out9, img9, grp10])
    btn10.click(step10_weights,                            outputs=[out10, grp11])
    btn11.click(step11_model,                              outputs=[out11, grp12])
    btn12.click(step12_train,                              outputs=[out12, img12, grp13])
    btn13.click(step13_loss_viz,                           outputs=[out13, img13, grp14])
    btn14.click(step14_evaluate,                           outputs=[out14, grp15])
    btn15.click(step15_confusion,                          outputs=[out15, img15a, img15b, grp16])
    btn16.click(step16_comparison,                         outputs=[out16, img16a, img16b])

if __name__ == "__main__":
    demo.queue()
    demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
